# Credit Card Fraud Challenge at Silverleaf National Bank

**DSE 200 · Day 3 · team challenge · class imbalance and feature engineering, for real**

## The situation

It's 9:15 AM at Silverleaf National Bank. The Head of Security has just told you fraud losses are up 47% this quarter. You have Jan to Sep 2024: about 75,000 card transactions, each labeled fraud or legitimate. About **1 in 90 is fraud.** You have Oct to Dec without labels. Flag the fraud.

## How you're scored

Not accuracy. Money. For every transaction in Oct to Dec:

| You say | It was | The bank |
| --- | --- | --- |
| approve | legit | **earns 2%** of the amount in fees |
| approve | fraud | **loses the full amount** |
| flag | legit | earns nothing (the customer is annoyed, the sale is lost) |
| flag | fraud | loses nothing |

The leaderboard ranks teams by **net dollars** over the quarter. Approving everything is a strategy; it loses money. Flagging everything earns exactly zero.

| Cell | What to do |
| --- | --- |
| `▶ SETUP` | Already written. Run it. |
| `✍️ YOU TRY` | Yours. Points in the heading. |
| `✅ CHECKPOINT` | Run it. It tells you whether your submission file is valid. |

In [ ]:
!uv pip install --system -q imbalanced-learn

In [ ]:
# ▶ SETUP — run first. Finds the data and defines the scoring rule.
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
    print("downloaded", name, "from the course repo")
    return name

train = pd.read_csv(find("fraud_training.csv"), parse_dates=["TransactionDate"])
test = pd.read_csv(find("fraud_test.csv"), parse_dates=["TransactionDate"])
print(f"train {train.shape}  {train.TransactionDate.min():%b %d} to {train.TransactionDate.max():%b %d}")
print(f"test  {test.shape}  {test.TransactionDate.min():%b %d} to {test.TransactionDate.max():%b %d}   (no IsFraud)")

FEE = 0.02   # the bank earns 2% of every legit transaction it approves

def net_dollars(amount, is_fraud, flagged):
    """What the bank makes over these transactions if it flags `flagged` (0/1 arrays)."""
    amount, is_fraud, flagged = map(np.asarray, (amount, is_fraud, flagged))
    earned = FEE * amount * ((flagged == 0) & (is_fraud == 0))
    lost = amount * ((flagged == 0) & (is_fraud == 1))
    return round(float((earned - lost).sum()), 2)

# Where each city is, for anyone who wants distances (8.3 has haversine_miles).
CITY_COORDS = {
    "New York": (40.7128, -74.0060), "Los Angeles": (34.0522, -118.2437),
    "Chicago": (41.8781, -87.6298), "Houston": (29.7604, -95.3698),
    "Phoenix": (33.4484, -112.0740), "Philadelphia": (39.9526, -75.1652),
    "San Antonio": (29.4241, -98.4936), "San Diego": (32.7157, -117.1611),
    "Dallas": (32.7767, -96.7970), "San Jose": (37.3382, -121.8863),
}
print("net_dollars() and CITY_COORDS ready.")

In [ ]:
# ▶ SETUP -- the two strategies that need no model
print("approve everything:", net_dollars(train.Amount, train.IsFraud, np.zeros(len(train))))
print("flag everything:   ", net_dollars(train.Amount, train.IsFraud, np.ones(len(train))))

# Part 1: Exploratory analysis (25 pts)

**Required (15 pts):**
1. How imbalanced is it? Fraud vs legit counts and share, and the imbalance ratio.
2. Amount for fraud vs legit (a box plot on a log axis reads better than a raw one).
3. Correlation among the numeric columns.

**Pick two (10 pts):** fraud by hour of day · by location · by transaction type · per customer (how many customers are ever hit?) · per merchant.

Under each plot, 2 to 3 sentences: what you see and what feature it suggests.

In [ ]:
# ✍️ YOU TRY -- 1. class balance

In [ ]:
# ✍️ YOU TRY -- 2. amount, fraud vs legit

In [ ]:
# ✍️ YOU TRY -- 3. correlation

*Your sentences.*

In [ ]:
# ✍️ YOU TRY -- two more

*Your sentences.*

# Part 2: Preprocessing and features (25 pts)

1. **Missing values and types (5 pts).** Check; say what you did, even if it was nothing.
2. **Encode categoricals (10 pts).** `TransactionType` and `Location`. One-hot, not label codes: San Diego isn't "7 more" than New York.
3. **Validation split (5 pts).** The test set is the *future*. Hold out the **last two months of training** (Aug and Sep) for validation, not a random 20%. A random split lets the model learn a customer's September fraud spree and "predict" it in August.
4. **Features (bonus, and where the points really are).** The raw columns don't know what's normal **for this customer**. Ideas: amount vs this customer's usual, a city that isn't home, minutes since their previous transaction, a jump between far cities faster than a plane, a merchant they've never used, tiny charges, amounts just under $1,000.

**The rule for every history feature:** compute it from the customer's *earlier* transactions only. The test rows can use the training rows as history (they come earlier in time); nothing may look at `IsFraud`.

In [ ]:
# ✍️ YOU TRY -- missing values, types

In [ ]:
# ✍️ YOU TRY -- features. Build them on train AND test together (sorted by time), then split back.

In [ ]:
# ✍️ YOU TRY -- validation split: Jan-Jul to fit, Aug-Sep to validate

# Part 3: Models and evaluation (25 pts)

1. **At least one model (15 pts).** Start with `LogisticRegression` (scale first). Then try a tree model.
2. **Plots:** ROC curve, **precision-recall curve** (at 1% fraud this one is more honest), confusion matrix, feature importance.
3. **Classification report** on validation.
4. **Look at recall and precision for fraud.** If the model barely flags anything, why? Fix it before Part 4: class weights, resampling (SMOTE on the fit rows only), or the threshold.
5. **Pick a threshold for money, not accuracy.** Use `net_dollars` on the validation months.

In [ ]:
# ✍️ YOU TRY -- models

*What does the table say?*

In [ ]:
# ✍️ YOU TRY -- plots: ROC, precision-recall, confusion matrix, feature importance

In [ ]:
# ✍️ YOU TRY -- threshold for money

*Which model and threshold, and why?*

# Part 4: Predictions (25 pts)

1. Build the same features for `test` (Part 2's function should already do it).
2. Refit your chosen model on **all** of the training months.
3. Write `fraud_predictions.csv` with exactly these columns: `TransactionID`, `IsFraud` (0/1, your decision), `FraudProbability` (your model's probability).
4. Comment the code so another team could rerun it.

Name the file `<team>_fraud_predictions.csv` when you upload it.

In [ ]:
# ✍️ YOU TRY -- refit, predict, save
submission = None   # TODO: a DataFrame with TransactionID, IsFraud, FraudProbability

In [ ]:
# ✅ CHECKPOINT -- is the file valid?
assert submission is not None, "build `submission` first"
assert list(submission.columns) == ["TransactionID", "IsFraud", "FraudProbability"], list(submission.columns)
assert len(submission) == len(test) and set(submission.TransactionID) == set(test.TransactionID), "one row per test transaction"
assert set(submission.IsFraud.unique()) <= {0, 1}
assert submission.FraudProbability.between(0, 1).all()
print(f"Valid. {submission.IsFraud.sum():,} of {len(submission):,} flagged ({submission.IsFraud.mean():.2%}).")

# Bonus (10 pts)

At least one:
1. **Feature engineering:** a feature nobody else on the board has, and its effect on validation net dollars.
2. **Model comparison:** three or more models in one table, same features, same validation months.
3. **Interactive dashboard:** Plotly (installed in Colab) showing fraud over time, by city, and your flags.

In [ ]:
# ✍️ YOU TRY -- bonus

## Wrap

- At 1% fraud, accuracy is useless and ROC flatters. Report recall, precision, and PR-AUC.
- Fraud is "unusual **for this customer**". The features that win say what's usual.
- History features use earlier rows only. The validation split respects time.
- Rebalancing moves the line. The cost table tells you where the line belongs.